# 06 — Raw Hazard EDA for Regulatory Agent Case-Study Selection

**Goal:** find a real, evidence-rich RASFF case-study pattern for the Regulatory Agent benchmark without letting the derived `Hazard_Type` decide the answer in advance.

### Analysis logic
`Origin × Category screening → raw Hazard inspection → missingness → conservative hazard normalization → Origin × Category × Normalized Hazard ranking → time/seriousness check → case-study shortlist`

### Main questions
1. Which `origin × category` pairs have enough repeated notifications?
2. For each strong pair, how complete is the original `hazards` field?
3. What exact/raw hazard strings are most frequently reported?
4. Are the same hazards fragmented across spelling/measurement variants?
5. After conservative normalization, which `origin × category × hazard` patterns dominate?
6. Are those patterns repeated across years?
7. Which 2–3 cases are strong enough for the Regulatory Agent benchmark and later EU-regulation verification?

> **Important:** `Hazard_Type` is not used to discover the candidate. It is shown only later as a descriptive cross-check. Notification counts describe RASFF notifications, not prevalence or causal risk.

In [ ]:
from pathlib import Path
import sys, re
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "rasff_core.py").exists():
    raise FileNotFoundError("Open this notebook from the RASFF_Rebuild folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rasff_core import load_prepared
df = load_prepared()

required = ["reference","date","origin","category","hazards","risk_decision","type"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Rows: {len(df):,}")
print(f"Date range: {df.date.min()} → {df.date.max()}")
display(df[["reference","date","origin","category","hazards","risk_decision"]].head())

## 1. Scope and raw-hazard quality

Start with food notifications with a known origin. We deliberately inspect the **original `hazards` field before using any derived hazard grouping**.

In [ ]:
eda = df.loc[(df["type"].str.lower() == "food") & df["origin"].ne("")].copy()
eda["year"] = eda["date"].dt.year
eda["hazard_raw"] = eda["hazards"].fillna("").astype(str).str.strip()
eda["hazard_missing"] = eda["hazard_raw"].eq("")

print(f"Food rows with known origin: {len(eda):,}")
print(f"Unique origins: {eda.origin.nunique():,}")
print(f"Unique categories: {eda.category.nunique():,}")
print(f"Missing raw hazards: {eda.hazard_missing.sum():,} ({eda.hazard_missing.mean():.1%})")
print(f"Unique non-empty raw hazard strings: {eda.loc[~eda.hazard_missing, 'hazard_raw'].nunique():,}")

## 2. Q1 — Which origin × category pairs are repeatedly reported?

This is the first screening layer. A high count is useful for case-study support, but **does not mean the country/product is more unsafe**.

In [ ]:
origin_category = (
    eda.groupby(["origin","category"])
       .agg(
           notifications=("reference","nunique"),
           years=("year","nunique"),
           first_date=("date","min"),
           last_date=("date","max"),
           hazard_missing_share=("hazard_missing","mean"),
           serious_share=("target","mean"),
       )
       .reset_index()
       .sort_values(["notifications","years"], ascending=False)
)

display(origin_category.head(30))

## 3. Q2 — Which pairs have enough usable raw hazard information?

A large pair with mostly blank hazards is weak for our benchmark. The thresholds below are **editable screening rules**, not scientific cut-offs.

In [ ]:
MIN_PAIR_NOTIFICATIONS = 30
MIN_YEARS = 3
MAX_HAZARD_MISSING_SHARE = 0.50

pair_candidates = origin_category.loc[
    (origin_category.notifications >= MIN_PAIR_NOTIFICATIONS) &
    (origin_category.years >= MIN_YEARS) &
    (origin_category.hazard_missing_share <= MAX_HAZARD_MISSING_SHARE)
].copy()

display(pair_candidates.head(25))
print(f"Eligible origin × category pairs: {len(pair_candidates)}")

## 4. Q3 — Inspect raw hazards before normalizing anything

Choose an `origin × category` pair from the table above. The default uses the highest-count eligible pair.

Look at the raw strings carefully. Ask:
- Is one recognizable hazard repeatedly reported?
- Is the same hazard split into measurement/spelling variants?
- Are entries compound descriptions containing more than one hazard?
- Is the field too inconsistent to normalize safely?

**Do not automatically merge scientifically different hazards just because the words look similar.**

In [ ]:
SELECTED_ORIGIN = pair_candidates.iloc[0]["origin"] if len(pair_candidates) else None
SELECTED_CATEGORY = pair_candidates.iloc[0]["category"] if len(pair_candidates) else None

pair = eda.loc[
    (eda.origin == SELECTED_ORIGIN) &
    (eda.category == SELECTED_CATEGORY)
].copy()

print("Selected:", SELECTED_ORIGIN, "|", SELECTED_CATEGORY)
print(f"Notifications: {pair.reference.nunique():,}")
print(f"Missing hazards: {pair.hazard_missing.mean():.1%}")

raw_counts = (
    pair.loc[~pair.hazard_missing]
        .groupby("hazard_raw")["reference"].nunique()
        .sort_values(ascending=False)
        .rename("notifications")
)
display(raw_counts.head(40))

display(
    pair.loc[~pair.hazard_missing, ["reference","date","hazard_raw","subject","risk_decision"]]
        .sort_values("date", ascending=False)
        .head(30)
)

## 5. Q4 — Create a **conservative** normalized hazard

The goal is not to recreate `Hazard_Type`. We want to preserve the actual hazard identity while removing superficial formatting differences.

The automatic function below only performs light text cleanup. It **does not** decide that, for example, all mycotoxins are the same hazard.

After seeing the raw counts, add only well-supported aliases to `MANUAL_HAZARD_ALIASES`. Keep this mapping small and auditable.

In [ ]:
def light_normalize_hazard(value):
    if pd.isna(value):
        return ""
    s = str(value).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*;\s*", "; ", s)
    return s

# Add aliases ONLY after inspecting raw strings.
# Exact cleaned string -> scientifically meaningful common label.
MANUAL_HAZARD_ALIASES = {
    # Example format only — uncomment/add after validation:
    # "aflatoxins": "aflatoxin",
    # "aflatoxin b1": "aflatoxin b1",
}

eda["hazard_clean"] = eda["hazard_raw"].map(light_normalize_hazard)
eda["hazard_normalized"] = eda["hazard_clean"].replace(MANUAL_HAZARD_ALIASES)

pair_norm = eda.loc[
    (eda.origin == SELECTED_ORIGIN) &
    (eda.category == SELECTED_CATEGORY) &
    eda.hazard_normalized.ne("")
].copy()

normalized_counts = (
    pair_norm.groupby("hazard_normalized")["reference"].nunique()
             .sort_values(ascending=False)
             .rename("notifications")
)
display(normalized_counts.head(40))

### Optional helper — find strings that may need manual review

This does **not** merge anything. It only shows frequent raw strings containing a keyword so you can inspect variants yourself.

In [ ]:
KEYWORD = ""  # e.g. "aflatoxin", "salmonella", "chlorpyrifos"

if KEYWORD:
    review = (
        eda.loc[eda.hazard_clean.str.contains(KEYWORD, case=False, na=False)]
           .groupby("hazard_clean")["reference"].nunique()
           .sort_values(ascending=False)
           .rename("notifications")
    )
    display(review.head(50))
else:
    print("Set KEYWORD after inspecting the raw hazard table if you want to review variants.")

## 6. Q5 — Rank origin × category × normalized hazard patterns

Now — and only now — combine the three dimensions.

Rows with missing hazards are excluded from this table, so we retain the pair-level missingness alongside each result.

In [ ]:
usable = eda.loc[eda.hazard_normalized.ne("")].copy()

three_way = (
    usable.groupby(["origin","category","hazard_normalized"])
          .agg(
              notifications=("reference","nunique"),
              years=("year","nunique"),
              first_date=("date","min"),
              last_date=("date","max"),
              serious_share=("target","mean"),
          )
          .reset_index()
)

pair_meta = origin_category[[
    "origin","category","notifications","hazard_missing_share"
]].rename(columns={"notifications":"origin_category_total"})

three_way = three_way.merge(pair_meta, on=["origin","category"], how="left")
three_way["share_within_origin_category"] = (
    three_way["notifications"] / three_way["origin_category_total"]
)

three_way = three_way.sort_values(
    ["notifications","years","share_within_origin_category"],
    ascending=False
)

display(three_way.head(40))

## 7. Q6 — Screen repeated exact-hazard candidates

We prefer patterns that have enough notifications, span multiple years, and represent a meaningful share of their origin/category pair.

Again, these thresholds are for **benchmark discovery**, not epidemiological inference.

In [ ]:
MIN_HAZARD_NOTIFICATIONS = 15
MIN_HAZARD_YEARS = 3
MIN_WITHIN_PAIR_SHARE = 0.10

hazard_candidates = three_way.loc[
    (three_way.notifications >= MIN_HAZARD_NOTIFICATIONS) &
    (three_way.years >= MIN_HAZARD_YEARS) &
    (three_way.share_within_origin_category >= MIN_WITHIN_PAIR_SHARE)
].copy()

display(hazard_candidates.head(30))
print(f"Exact-hazard candidate patterns: {len(hazard_candidates)}")

## 8. Q7 — Is the selected hazard pattern persistent over time?

Pick one candidate and inspect annual counts. A pattern concentrated in one unusual year may still be interesting, but it should not be described as a stable recurring pattern.

In [ ]:
SELECTED_HAZARD = (
    hazard_candidates.iloc[0]["hazard_normalized"]
    if len(hazard_candidates) else None
)

case = eda.loc[
    (eda.origin == SELECTED_ORIGIN) &
    (eda.category == SELECTED_CATEGORY) &
    (eda.hazard_normalized == SELECTED_HAZARD)
].copy()

print("Case:", SELECTED_ORIGIN, "|", SELECTED_CATEGORY, "|", SELECTED_HAZARD)

yearly = (
    case.groupby("year")["reference"].nunique()
        .rename("notifications")
        .reset_index()
)
display(yearly)

if len(yearly):
    ax = yearly.plot(
        x="year", y="notifications", kind="bar",
        legend=False, figsize=(8,4),
        title="Selected raw-hazard pattern by year"
    )
    ax.set_ylabel("RASFF notifications")
    plt.tight_layout()
    plt.show()

## 9. Q8 — Bring `Hazard_Type` back only as a cross-check

At this stage we can inspect whether the old derived `Hazard_Type` labels the selected exact hazard consistently. It is **context**, not the basis of candidate discovery.

In [ ]:
if "Hazard_Type" in case.columns:
    display(
        case.groupby("Hazard_Type")["reference"].nunique()
            .sort_values(ascending=False)
            .rename("notifications")
    )
else:
    print("Hazard_Type is not present — no cross-check needed.")

## 10. Q9 — Build a shortlist for Regulatory Agent feasibility checking

Do **not** select a winner automatically. The next step requires human/domain review and official regulatory-source checking.

Review:
- **Data support:** enough notifications and years?
- **Raw-data quality:** acceptable hazard completeness?
- **Hazard clarity:** does the normalized label represent a real, specific hazard rather than a broad class?
- **Regulatory link:** can it connect to identifiable EU legislation or official guidance?
- **RAG value:** is there regulatory evidence worth retrieving?
- **Tool value:** does answering benefit from RASFF search/data analysis?
- **Agent value:** can the case progress from lookup → evidence → analysis → multi-step investigation?
- **Interpretability:** can we avoid implying notification count = prevalence/risk?

Shortlist **2–3 candidates** for official-source verification before freezing the 10-question benchmark.

In [ ]:
shortlist = hazard_candidates.head(20).copy()

# Optional descriptive context only
if "Hazard_Type" in eda.columns:
    type_mode = (
        usable.groupby(["origin","category","hazard_normalized"])["Hazard_Type"]
              .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "")
              .rename("derived_hazard_type")
              .reset_index()
    )
    shortlist = shortlist.merge(
        type_mode,
        on=["origin","category","hazard_normalized"],
        how="left"
    )

shortlist["raw_hazard_reviewed"] = False
shortlist["regulatory_link_checked"] = False
shortlist["notes"] = ""

display(shortlist)

out = ROOT / "artifacts" / "regulatory_agent_raw_hazard_candidates.csv"
shortlist.to_csv(out, index=False)
print("Saved:", out)

## Interpretation guardrails

1. **Notification count ≠ prevalence or true risk.** RASFF records notified events, not all products, imports, inspections, or exposures.
2. **Origin ≠ causation.** Trade volume, border-control intensity, targeted sampling and reporting practice can affect counts.
3. **Raw `hazards` is the primary field for this EDA.** `Hazard_Type` is derived and used only after candidate discovery.
4. **Normalization must be conservative and auditable.** Do not merge scientifically different hazards for convenience.
5. **Missing hazards matter.** A blank field does not mean that no hazard existed.
6. **One RASFF notification is the counting unit.** We use unique `reference` counts to avoid accidental row duplication.
7. **This is case-study selection, not causal inference or model training.**

### What to bring to the next step

After running the notebook, share:
- the top 10–20 rows of `pair_candidates`,
- the top raw-hazard table for 2–3 interesting origin/category pairs,
- any manual aliases you think are scientifically justified,
- the top 10–20 rows of `hazard_candidates`.

Then we can verify official EU regulatory evidence and freeze the benchmark scenario.